# 6. ETL — Punjenje dimenzijskog modela

Ovaj notebook izvršava ETL (Extract, Transform, Load) proces iz **dva izvora podataka**:

- **Izvor 1:** `Support_tickets_PROCESSED.csv` — predprocesirani 80% skup (53.353 × 42 stupca)
- **Izvor 2:** `Support_tickets_named.csv` — originalni puni dataset (66.691 × 58 stupaca)
  - sadrži 16 rijetkih `wf_*` stupaca koji su izbačeni predprocesiranjem (>90% NULL)
  - koriste se za obogaćivanje fact tablice s dodatnim workflow vremenima

### Koraci:
1. Pražnjenje postojećih podataka u dimenzijskim tablicama
2. Punjenje dimenzija (`dim_tehnicar`, `dim_projekt_prioritet_status`, `dim_tip_ticketa`, `dim_rezolucija`, `dim_vrijeme`)
3. Transformacija metrika iz oba izvora (sekunde → sati)
4. Punjenje tablice činjenica (`fact_support_tickets`)

**Preduvjeti:** Pokrenuti notebook `5_dimensional_ddl.ipynb` za kreiranje tablica.

In [1]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

DB_USER = os.getenv('DB_USER', 'root')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_NAME = os.getenv('DB_NAME', 'fipu_srp_projekt')

if not DB_PASSWORD:
    raise ValueError("DB_PASSWORD nije postavljen! Kreiraj .env datoteku.")

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")
print(f"Spojeno na bazu: {DB_NAME} ({DB_HOST})")

Spojeno na bazu: fipu_srp_projekt (localhost)


## 6.1 Pražnjenje tablica

In [2]:
with engine.connect() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    conn.execute(text("TRUNCATE TABLE fact_support_tickets;"))
    conn.execute(text("TRUNCATE TABLE dim_vrijeme;"))
    conn.execute(text("TRUNCATE TABLE dim_tehnicar;"))
    conn.execute(text("TRUNCATE TABLE dim_projekt_prioritet_status;"))
    conn.execute(text("TRUNCATE TABLE dim_tip_ticketa;"))
    conn.execute(text("TRUNCATE TABLE dim_rezolucija;"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))
    conn.commit()
    print("Tablice ispražnjene.")

Tablice ispražnjene.


## 6.2 EXTRACT — Učitavanje podataka iz dva izvora

In [3]:
# IZVOR 1: Support_tickets_PROCESSED.csv — predprocesirani 80% skup
df = pd.read_csv('Support_tickets_PROCESSED.csv')
df['issue_created'] = pd.to_datetime(df['issue_created'], format='ISO8601')
df['issue_resolution_date'] = pd.to_datetime(df['issue_resolution_date'], format='ISO8601')
print(f"Izvor 1 ucitan: {len(df):,} redaka, {len(df.columns)} stupaca")
print(f"Raspon: {df['issue_created'].min()} -- {df['issue_created'].max()}")
print()

# IZVOR 2: Support_tickets_named.csv — originalni puni dataset
# Sadrzi 16 rijetkih wf_* stupaca koji su izbaceni predprocesiranjem (>90% NULL)
# ali sadrze stvarne podatke za podskup ticketa (wf_validation: 2.961, wf_reopened: 2.123...)
extra_wf_cols = [
    'id', 'wf_validation', 'wf_reopened', 'wf_resolved_under_monitoring',
    'wf_approved', 'wf_closed', 'wf_pending_deployment', 'wf_done',
    'wf_monitoring', 'wf_under_review', 'wf_to_do', 'wf_deployment',
    'wf_cancelled', 'wf_in_review', 'wf_pending_customer_approval',
    'wf_rejected', 'wf_testing_monitoring'
]
df_named = pd.read_csv('Support_tickets_named.csv', usecols=extra_wf_cols)

# Pretvorba sekunde -> sati za dodatne wf_ stupce
for col in extra_wf_cols[1:]:
    df_named[col] = (df_named[col] / 3600).round(2)
df_named.columns = ['id'] + ['sati_' + c[3:] for c in extra_wf_cols[1:]]

print(f"Izvor 2 ucitan: {len(df_named):,} redaka, {len(extra_wf_cols)-1} dodatnih wf_ stupaca")
non_null_total = df_named.iloc[:, 1:].notna().sum().sum()
print(f"Non-null vrijednosti u dodatnim stupcima: {non_null_total:,}")
print()

# Spajanje Izvora 1 i Izvora 2 po ID-u (LEFT JOIN — cuvamo sve 53.353 ticketa)
df_named['id'] = df_named['id'].astype('Int64')
df = df.merge(df_named, on='id', how='left')
print(f"Nakon spajanja izvora: {len(df):,} redaka, {len(df.columns)} stupaca")
with_extra = df.iloc[:, -16:].notna().any(axis=1).sum()
print(f"Ticketi s barem jednim dodatnim wf_ podatkom: {with_extra:,}")

Izvor 1 ucitan: 53,353 redaka, 42 stupaca
Raspon: 2007-04-15 08:12:11+00:00 -- 2023-03-14 14:00:32+00:00

Izvor 2 ucitan: 66,691 redaka, 16 dodatnih wf_ stupaca
Non-null vrijednosti u dodatnim stupcima: 16,665

Nakon spajanja izvora: 53,353 redaka, 58 stupaca
Ticketi s barem jednim dodatnim wf_ podatkom: 8,441


## 6.3 LOAD — Punjenje dimenzija

In [4]:
# DIM_TEHNICAR
reporters = df[['issue_reporter']].rename(columns={'issue_reporter': 'ime_prezime'})
assignees = df[['issue_assignee']].rename(columns={'issue_assignee': 'ime_prezime'})
dim_tehnicar = pd.concat([reporters, assignees]).drop_duplicates().dropna().reset_index(drop=True)
dim_tehnicar.to_sql('dim_tehnicar', engine, if_exists='append', index=False)
db_tehnicari = pd.read_sql("SELECT tehnicar_key, ime_prezime FROM dim_tehnicar", engine)
print(f"dim_tehnicar: {len(dim_tehnicar)} redaka")

# DIM_PROJEKT_PRIORITET_STATUS (junk dimenzija)
dim_junk = df[['issue_proj', 'issue_priority', 'issue_status']].drop_duplicates().reset_index(drop=True)
dim_junk.columns = ['naziv_projekta', 'razina_prioriteta', 'naziv_statusa']
dim_junk.to_sql('dim_projekt_prioritet_status', engine, if_exists='append', index=False)
db_junk = pd.read_sql(
    "SELECT projekt_prioritet_status_key, naziv_projekta, razina_prioriteta, naziv_statusa FROM dim_projekt_prioritet_status",
    engine
)
print(f"dim_projekt_prioritet_status: {len(dim_junk)} redaka (junk dimenzija)")

# DIM_TIP_TICKETA
dim_tip = df[['issue_type']].drop_duplicates().dropna().reset_index(drop=True)
dim_tip.columns = ['naziv_tipa']
dim_tip.to_sql('dim_tip_ticketa', engine, if_exists='append', index=False)
db_tip = pd.read_sql("SELECT tip_ticketa_key, naziv_tipa FROM dim_tip_ticketa", engine)
print(f"dim_tip_ticketa: {len(dim_tip)} redaka")

# DIM_REZOLUCIJA
dim_rez = df[['issue_resolution']].drop_duplicates().dropna().reset_index(drop=True)
dim_rez.columns = ['naziv_rezolucije']
dim_rez.to_sql('dim_rezolucija', engine, if_exists='append', index=False)
db_rez = pd.read_sql("SELECT rezolucija_key, naziv_rezolucije FROM dim_rezolucija", engine)
print(f"dim_rezolucija: {len(dim_rez)} redaka")

# DIM_VRIJEME
dates = df['issue_created'].dt.date.unique()
dim_vrijeme = pd.DataFrame({'vrijeme_key': pd.to_datetime(dates)})
dim_vrijeme['dan'] = dim_vrijeme['vrijeme_key'].dt.day
dim_vrijeme['mjesec'] = dim_vrijeme['vrijeme_key'].dt.month
dim_vrijeme['godina'] = dim_vrijeme['vrijeme_key'].dt.year
dim_vrijeme['kvartal'] = dim_vrijeme['vrijeme_key'].dt.quarter
dim_vrijeme['dan_u_tjednu'] = dim_vrijeme['vrijeme_key'].dt.day_name()
dim_vrijeme.to_sql('dim_vrijeme', engine, if_exists='append', index=False)
print(f"dim_vrijeme: {len(dim_vrijeme)} redaka")

dim_tehnicar: 100 redaka
dim_projekt_prioritet_status: 353 redaka (junk dimenzija)
dim_tip_ticketa: 15 redaka
dim_rezolucija: 4 redaka
dim_vrijeme: 4706 redaka


## 6.4 TRANSFORM — Izračun metrika (Izvor 1 + Izvor 2)

In [5]:
# IZVOR 1: Ukupno vrijeme rjesavanja (sati)
df['vrijeme_rjesavanja_sati'] = (
    (df['issue_resolution_date'] - df['issue_created']).dt.total_seconds() / 3600
)
df.loc[df['vrijeme_rjesavanja_sati'] < 0, 'vrijeme_rjesavanja_sati'] = 0
df['vrijeme_rjesavanja_sati'] = df['vrijeme_rjesavanja_sati'].round(2)

# IZVOR 1: Glavna 4 workflow stanja (sekunde -> sati)
df['sati_open'] = (df['wf_open'] / 3600).round(2)
df['sati_in_progress'] = (df['wf_in_progress'] / 3600).round(2)
df['sati_resolved'] = (df['wf_resolved'] / 3600).round(2)
df['sati_waiting'] = (df['wf_waiting'] / 3600).round(2)

print("Metrike iz Izvora 1 (Support_tickets_PROCESSED.csv):")
for col in ['vrijeme_rjesavanja_sati', 'sati_open', 'sati_in_progress', 'sati_resolved', 'sati_waiting']:
    print(f"  {col:30s} prosjek: {df[col].mean():10.1f} h, NULL: {df[col].isna().sum()}")
print()

# IZVOR 2: Dodatni workflow stupci iz Support_tickets_named.csv
# Vec su pretvoreni u sate i spojeni u koraku 6.2 (EXTRACT)
extra_sati = [c for c in df.columns if c.startswith('sati_')
              and c not in ['sati_open','sati_in_progress','sati_resolved','sati_waiting']]
print(f"Metrike iz Izvora 2 (Support_tickets_named.csv) — {len(extra_sati)} dodatnih stanja:")
for col in sorted(extra_sati):
    non_null = df[col].notna().sum()
    if non_null > 0:
        print(f"  {col:35s} non-null: {non_null:>5,}  prosjek: {df[col].mean():8.1f} h")

Metrike iz Izvora 1 (Support_tickets_PROCESSED.csv):
  vrijeme_rjesavanja_sati        prosjek:    18441.0 h, NULL: 698
  sati_open                      prosjek:    17819.5 h, NULL: 60
  sati_in_progress               prosjek:      266.8 h, NULL: 23918
  sati_resolved                  prosjek:       86.6 h, NULL: 30795
  sati_waiting                   prosjek:      481.5 h, NULL: 37466

Metrike iz Izvora 2 (Support_tickets_named.csv) — 16 dodatnih stanja:
  sati_approved                       non-null:   680  prosjek:   2723.8 h
  sati_cancelled                      non-null: 1,269  prosjek:    388.0 h
  sati_closed                         non-null:   215  prosjek:    137.4 h
  sati_deployment                     non-null: 1,547  prosjek:    312.5 h
  sati_done                           non-null:   241  prosjek:    346.9 h
  sati_in_review                      non-null:   132  prosjek:     95.5 h
  sati_monitoring                     non-null: 1,683  prosjek:     50.5 h
  sati_pending_c

## 6.5 TRANSFORM — Mapiranje stranih ključeva

In [6]:
fact = df.copy()

# Reporter (inner join — svaki ticket ima reportera)
fact = fact.merge(db_tehnicari, left_on='issue_reporter', right_on='ime_prezime', how='inner')
fact = fact.rename(columns={'tehnicar_key': 'reporter_key'})
print(f"Nakon merge s dim_tehnicar (reporter): {len(fact)} redaka")

# Assignee (LEFT join — 46% ticketa nema assignee)
fact = fact.merge(db_tehnicari, left_on='issue_assignee', right_on='ime_prezime',
                  how='left', suffixes=('', '_assignee'))
fact = fact.rename(columns={'tehnicar_key': 'assignee_key'})
print(f"Nakon merge s dim_tehnicar (assignee): {len(fact)} redaka (NULL: {fact['assignee_key'].isna().sum()})")

# Junk dimenzija (inner join na 3 stupca)
fact = fact.merge(db_junk,
                  left_on=['issue_proj', 'issue_priority', 'issue_status'],
                  right_on=['naziv_projekta', 'razina_prioriteta', 'naziv_statusa'],
                  how='inner')
print(f"Nakon merge s dim_projekt_prioritet_status: {len(fact)} redaka")

# Tip ticketa (inner join)
fact = fact.merge(db_tip, left_on='issue_type', right_on='naziv_tipa', how='inner')
print(f"Nakon merge s dim_tip_ticketa: {len(fact)} redaka")

# Rezolucija (LEFT join — 1.3% ticketa nema rezoluciju)
fact = fact.merge(db_rez, left_on='issue_resolution', right_on='naziv_rezolucije',
                  how='left', suffixes=('', '_rez'))
print(f"Nakon merge s dim_rezolucija: {len(fact)} redaka (NULL: {fact['rezolucija_key'].isna().sum()})")

Nakon merge s dim_tehnicar (reporter): 53353 redaka
Nakon merge s dim_tehnicar (assignee): 53353 redaka (NULL: 24771)
Nakon merge s dim_projekt_prioritet_status: 53353 redaka
Nakon merge s dim_tip_ticketa: 53353 redaka
Nakon merge s dim_rezolucija: 53353 redaka (NULL: 698)


## 6.6 LOAD — Punjenje fact tablice

In [7]:
# Priprema finalnog DataFrame-a prema DDL shemi
fact['vrijeme_key'] = fact['issue_created'].dt.date

fact_final = fact[[
    'id', 'reporter_key', 'assignee_key',
    'projekt_prioritet_status_key', 'tip_ticketa_key', 'rezolucija_key', 'vrijeme_key',
    'vrijeme_rjesavanja_sati', 'issue_comments_count',
    'sati_open', 'sati_in_progress', 'sati_resolved', 'sati_waiting'
]].copy()

fact_final = fact_final.rename(columns={
    'id': 'ticket_id',
    'issue_comments_count': 'broj_komentara'
})
fact_final['ticket_id'] = fact_final['ticket_id'].astype(int)
fact_final['assignee_key'] = fact_final['assignee_key'].astype('Int64')
fact_final['rezolucija_key'] = fact_final['rezolucija_key'].astype('Int64')

# Bulk insert
with engine.connect() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    conn.commit()

fact_final.to_sql('fact_support_tickets', engine, if_exists='append', index=False, chunksize=5000)

with engine.connect() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))
    conn.commit()

print(f"ETL uspješno završen! Uneseno {len(fact_final)} redaka u fact_support_tickets.")

ETL uspješno završen! Uneseno 53353 redaka u fact_support_tickets.


## 6.7 Verifikacija ETL procesa

In [8]:
print("=" * 55)
print("VERIFIKACIJA ETL PROCESA")
print("=" * 55)
print()
print("IZVORI:")
print(f"  Izvor 1 (PROCESSED.csv):  53.353 redaka, 42 stupca")
print(f"  Izvor 2 (named.csv):      16 dodatnih wf_* stupaca")
print()
print("DIMENZIJE I FACT TABLICA:")
with engine.connect() as conn:
    for tbl in ['dim_tehnicar', 'dim_projekt_prioritet_status',
                'dim_tip_ticketa', 'dim_rezolucija', 'dim_vrijeme',
                'fact_support_tickets']:
        cnt = conn.execute(text(f"SELECT COUNT(*) FROM {tbl}")).scalar()
        print(f"  {tbl:40s} {cnt:>6,} redaka")

print(f"\nOcekivano u fact tablici: {len(fact_final):,}")
print()

# Provjera nema izgubljenih podataka
assert len(fact_final) == 53353, f"GRESKA: ocekivano 53353, dobiveno {len(fact_final)}"
print("Provjera izgubljenih podataka: PASS (0 izgubljenih redaka)")

sample = pd.read_sql("""
    SELECT
        f.ticket_id,
        j.naziv_projekta AS projekt,
        r.ime_prezime AS reporter,
        a.ime_prezime AS assignee,
        j.razina_prioriteta AS prioritet,
        j.naziv_statusa AS status,
        t.naziv_tipa AS tip_ticketa,
        rez.naziv_rezolucije AS rezolucija,
        f.vrijeme_key,
        f.vrijeme_rjesavanja_sati,
        f.broj_komentara
    FROM fact_support_tickets f
    JOIN dim_tehnicar r ON f.reporter_key = r.tehnicar_key
    LEFT JOIN dim_tehnicar a ON f.assignee_key = a.tehnicar_key
    JOIN dim_projekt_prioritet_status j ON f.projekt_prioritet_status_key = j.projekt_prioritet_status_key
    JOIN dim_tip_ticketa t ON f.tip_ticketa_key = t.tip_ticketa_key
    LEFT JOIN dim_rezolucija rez ON f.rezolucija_key = rez.rezolucija_key
    LIMIT 10
""", engine)
print("Uzorak iz star sheme (10 redaka):")
print(sample.to_string(index=False))

VERIFIKACIJA ETL PROCESA

IZVORI:
  Izvor 1 (PROCESSED.csv):  53.353 redaka, 42 stupca
  Izvor 2 (named.csv):      16 dodatnih wf_* stupaca

DIMENZIJE I FACT TABLICA:
  dim_tehnicar                                100 redaka
  dim_projekt_prioritet_status                353 redaka
  dim_tip_ticketa                              15 redaka
  dim_rezolucija                                4 redaka
  dim_vrijeme                               4,706 redaka
  fact_support_tickets                     53,353 redaka

Ocekivano u fact tablici: 53,353

Provjera izgubljenih podataka: PASS (0 izgubljenih redaka)
Uzorak iz star sheme (10 redaka):
 ticket_id         projekt       reporter       assignee prioritet status tip_ticketa       rezolucija vrijeme_key  vrijeme_rjesavanja_sati  broj_komentara
     10616    Project Nova Petar Pavlović    Josip Jurić   unknown closed      Ticket             Done  2016-03-27                     5.88               2
     10617  Project Aurora  Luka Pavlović     Luka 